In [1]:
import psycopg2
from psycopg2.extras import execute_values
from fastapi import FastAPI
from datetime import datetime, timedelta, date
from typing import Optional, List, Dict, Any
from pydantic import BaseModel
import json
from decimal import Decimal
from fastapi.middleware.cors import CORSMiddleware


# ---------- Database Configuration ----------
DB_HOST = "aws-1-us-east-2.pooler.supabase.com"
DB_NAME = "postgres"
DB_USER = "postgres.fdcwkvaivhrokkotmzwf"
DB_PASSWORD = "CS8803desover"
DB_PORT = 5432

def get_db_connection():
    return psycopg2.connect(
        host=DB_HOST,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT
    )

In [5]:
from score_trips import score_all_trips, load_data, preprocess_trips

trips, users, friendships = load_data()
trips = preprocess_trips(trips)
scores = score_all_trips(trips, friendships)
scores = scores.reset_index(drop=True)
scores['recommendation_id'] = scores.index

Loaded 2,181 trips across 35 users
Loaded 35 user profiles
Loaded 69 friendship edges
Scoring trips for user 0...
Scoring trips for user 1...
Scoring trips for user 2...
Scoring trips for user 3...
Scoring trips for user 4...
Scoring trips for user 5...
Scoring trips for user 6...
Scoring trips for user 7...
Scoring trips for user 8...
Scoring trips for user 9...
Scoring trips for user 10...
Scoring trips for user 11...
Scoring trips for user 12...
Scoring trips for user 13...
Scoring trips for user 14...
Scoring trips for user 15...
Scoring trips for user 16...
Scoring trips for user 17...
Scoring trips for user 18...
Scoring trips for user 19...
Scoring trips for user 20...
Scoring trips for user 21...
Scoring trips for user 22...
Scoring trips for user 23...
Scoring trips for user 24...
Scoring trips for user 25...
Scoring trips for user 26...
Scoring trips for user 27...
Scoring trips for user 28...
Scoring trips for user 29...
Scoring trips for user 30...
Scoring trips for user 31

In [3]:
print(f"Found {len(scores)} matches between recurring trips of friends")
scores

Found 33576 matches between recurring trips of friends


,user_id,friend_user_id,routine_id,friend_routine_id,match_score,co2_saved_kg,mode,details,recommendation_id
0,0,17,390,429,0.668592,4.085499,carpool,"{'scenario': 'D (enroute→enroute)', 'pickup_be...",0
1,0,17,161,200,0.626383,4.085499,carpool,"{'scenario': 'D (enroute→enroute)', 'pickup_be...",1
2,0,17,1,40,0.560509,4.085499,carpool,"{'scenario': 'D (enroute→enroute)', 'pickup_be...",2
3,0,17,1650,1690,0.559424,4.085499,carpool,"{'scenario': 'D (enroute→enroute)', 'pickup_be...",3
4,0,17,552,592,0.516815,4.085499,carpool,"{'scenario': 'D (enroute→enroute)', 'pickup_be...",4
...,...,...,...,...,...,...,...,...,...
33571,28,9,1717,1126,0.150936,-0.986868,carpool,"{'scenario': 'B (enroute→end)', 'pickup_begin_...",33571
33572,28,9,1327,1126,0.150424,-0.986868,carpool,"{'scenario': 'B (enroute→end)', 'pickup_begin_...",33572
33573,28,9,1248,1126,0.150033,-0.986868,carpool,"{'scenario': 'B (enroute→end)', 'pickup_begin_...",33573
33574,28,9,1089,1126,0.149477,-0.986868,carpool,"{'scenario': 'B (enroute→end)', 'pickup_begin_...",33574


In [4]:
columns = [
        'recommendation_id', 'user_id', 'mode', 'friend_user_id',
        'friend_routine_id', 'routine_id', 'match_score', 'co2_saved_kg', 'details'
    ]

df = scores.copy()
df['details'] = df['details'].apply(
    lambda x: json.dumps(x) if isinstance(x, dict) else x
)

rows = [tuple(row) for row in df[columns].itertuples(index=False, name=None)]

sql = f"""
    INSERT INTO recommendations ({', '.join(columns)})
    VALUES %s
    ON CONFLICT (recommendation_id) DO UPDATE SET
        user_id            = EXCLUDED.user_id,
        mode               = EXCLUDED.mode,
        friend_user_id     = EXCLUDED.friend_user_id,
        friend_routine_id  = EXCLUDED.friend_routine_id,
        routine_id         = EXCLUDED.routine_id,
        match_score        = EXCLUDED.match_score,
        co2_saved_kg       = EXCLUDED.co2_saved_kg,
        details            = EXCLUDED.details
"""

conn = get_db_connection()
try:
    with conn.cursor() as cur:
        execute_values(cur, sql, rows)
    conn.commit()
    print(f"Upserted {len(rows)} recommendations.")
except Exception as e:
    conn.rollback()
    raise e
finally:
    conn.close()

Upserted 33576 recommendations.


In [6]:
trips['user_id'].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34])

In [15]:
from recurring_trips import get_recurring_trips
import pandas as pd

routines = None
for user_id in trips['user_id'].unique():
    user_trips = trips[trips['user_id'] == user_id]
    user_routines = get_recurring_trips(user_trips, min_occurrences=2)
    user_routines = user_routines[['trip_id', 'user_id', 'start_cluster', 'end_cluster', 'occurrences']]
    user_routines = user_routines.rename(columns={'trip_id': 'routine_id'})

    if routines is None:
        routines = user_routines
    else:
        routines = pd.concat([routines, user_routines], ignore_index=True)

In [16]:
routines

,routine_id,user_id,start_cluster,end_cluster,occurrences
0,0,0,0,0,20
1,1,0,1,1,13
2,80,0,0,0,20
3,161,0,0,0,20
4,162,0,1,1,13
...,...,...,...,...,...
1203,1885,34,0,0,20
1204,1886,34,1,1,12
1205,2096,34,0,0,20
1206,2179,34,0,0,20


In [22]:
columns = [
        'routine_id', 'user_id', 'start_cluster', 'end_cluster', 'occurrences'
    ]

df = routines.copy()

rows = [tuple(row) for row in df[columns].itertuples(index=False, name=None)]

sql = f"""
    INSERT INTO routines ({', '.join(columns)})
    VALUES %s
    ON CONFLICT (routine_id) DO UPDATE SET
        user_id            = EXCLUDED.user_id,
        start_cluster      = EXCLUDED.start_cluster,
        end_cluster        = EXCLUDED.end_cluster,
        occurrences        = EXCLUDED.occurrences
"""

conn = get_db_connection()
try:
    with conn.cursor() as cur:
        execute_values(cur, sql, rows)
    conn.commit()
    print(f"Upserted {len(rows)} routines.")
except Exception as e:
    conn.rollback()
    raise e
finally:
    conn.close()

Upserted 1208 routines.
